# Capítulo 5 · Actividad: Titanic, ¿árbol o vecinos?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/04_actividad_titanic_arbol_knn.ipynb)

Compara un **árbol de decisión** y **KNN** para predecir si un pasajero del Titanic sobrevivió. Primero entiende los datos; el preprocesamiento lo decides tú. Los dos modelos tienen que evaluarse sobre la misma división de train y test, y con las mismas particiones de validación cruzada.

Cada fila es un pasajero. Para esta actividad usa las siguientes variables:

| Variable | Significado |
|---|---|
| `survived` | **Target**: `0` = no sobrevivió, `1` = sobrevivió |
| `pclass` | Clase del boleto: primera, segunda o tercera |
| `sex` | Sexo del pasajero |
| `age` | Edad en años |
| `sibsp` | Número de hermanos y cónyuges a bordo |
| `parch` | Número de padres e hijos a bordo |
| `fare` | Tarifa pagada |
| `embarked` | Puerto de embarque: `C`, `Q` o `S` |

Hay valores faltantes en `age` y `embarked`. Las clases de supervivencia no están balanceadas y las variables numéricas tienen escalas distintas.

**Usa únicamente las siete variables predictoras de la tabla.** El CSV contiene otras columnas: `alive`, por ejemplo, expresa la misma respuesta que `survived`. Incluirla entre las entradas sería una fuga de información: el modelo recibiría la respuesta que debe predecir. Tampoco incluyas `survived` entre las entradas.

Como mínimo, tu notebook debe incluir:

- La exploración de antes de modelar: forma de la tabla, nulos, balance de supervivencia, distribuciones y relación de las variables con la supervivencia.
- Una división estratificada de train y test y validación cruzada estratificada sobre train. Usa `random_state=42` en la división, en el árbol y al mezclar las particiones de validación.
- Un **árbol de decisión** y un **KNN**, los dos para clasificar. Decide cómo tratar los nulos, cómo codificar las variables categóricas y si estandarizas las numéricas: a KNN la escala le importa, al árbol no. Integra el preprocesamiento en un `Pipeline` para que se ajuste solo con los datos de entrenamiento de cada partición.
- La profundidad del árbol (y `min_samples_leaf`, si quieres) elegida con `GridSearchCV` sobre la validación cruzada.
- Para KNN, varios valores de `k`, incluyendo `k=1`, una **gráfica de la accuracy de train y de validación contra `k`** y el `k` que alcanza el máximo en validación. Distingue la accuracy sobre los mismos datos usados para entrenar de la obtenida en las particiones de validación. El test se usa una vez, al final, para evaluar ambos modelos elegidos, no para escoger hiperparámetros.
- Una **tabla** con los dos modelos: accuracy en validación cruzada (media y desviación estándar) y en prueba, y la matriz de confusión de cada uno en prueba. Compara también con una referencia sencilla: predecir siempre la clase mayoritaria de train.
- Tus **conclusiones**: con cuál te quedas y por qué. Explica la diferencia entre train y validación con `k=1`, cómo cambia al aumentar `k` y qué errores comete cada modelo al predecir supervivencia.

No hay una accuracy objetivo ni un ganador obligatorio. El objetivo es justificar la comparación con los resultados: `k=1` puede generalizar bien y la curva de validación no tiene por qué mejorar siempre al aumentar `k`. Si los modelos obtienen resultados muy parecidos, reconócelo en tus conclusiones.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

## Qué vamos a hacer

Comparamos un árbol de decisión y un KNN con las mismas siete variables, la misma división de train y test y las mismas particiones de validación cruzada. Así cualquier diferencia entre los dos viene del modelo y no de los datos con los que se probó.

Hay tres decisiones que se toman con evidencia y solo con train:

1. **Escalar o no.** Un KNN mide distancias entre pasajeros, así que la escala de cada columna le importa.
2. **La profundidad del árbol**, que controla cuántas preguntas hace antes de decidir.
3. **El valor de `k`**, el número de vecinos que votan en KNN.

Además de usar `KNeighborsClassifier`, calculamos a mano cómo decide para entender qué hace por dentro. El test se usa una sola vez, en la sección 7, y en la sección 8 repetimos la comparación con otros repartos para ver si el resultado fue casualidad.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

# Que scikit-learn devuelva DataFrames (con nombres de columna)
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
colores = {"no sobrevivió": "#4c72b0", "sobrevivió": "#dd8452"}

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "titanic.csv")
df.head()

## 1. Exploración de los datos

Primero miramos cómo vienen los datos. En esta sección solo describimos: las decisiones se toman después, con train.

In [ ]:
VARIABLES = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]

print("Filas y columnas del archivo:", df.shape)
resumen = pd.DataFrame({
    "tipo": df[VARIABLES + ["survived"]].dtypes.astype(str),
    "nulos": df[VARIABLES + ["survived"]].isna().sum(),
    "% de nulos": (df[VARIABLES + ["survived"]].isna().mean() * 100).round(1),
})
display(resumen)

print("Filas duplicadas en las 7 variables y el target:", df[VARIABLES + ["survived"]].duplicated().sum())
print()
print("¿Qué pasa con `alive`? Comparada con `survived`:")
display(pd.crosstab(df["survived"], df["alive"]))

**Qué vemos.** El archivo tiene 891 pasajeros y las siete variables predictoras casi completas. Solo faltan la edad de 177 pasajeros (19.9%) y el puerto de embarque de 2 (0.2%); en el resto no falta nada. Más adelante el `Pipeline` rellena esos huecos solo con lo que aprende de train.

Hay 111 filas repetidas en las siete variables y el target. No las quitamos: son pasajeros distintos que coinciden en todos sus valores, no un error de captura. Más adelante veremos que importan para KNN.

La tabla de `alive` contra `survived` muestra por qué la consigna pide no usarla: los 549 que no sobrevivieron aparecen con `no` y los 342 que sí, con `yes`, sin una sola excepción. Es la misma respuesta escrita de otra forma, y si entrara como variable el modelo acertaría sin aprender nada. Tampoco usamos `class`, `who`, `deck` ni las demás columnas extra del CSV.

In [ ]:
etiqueta = df["survived"].map({0: "no sobrevivió", 1: "sobrevivió"})
conteo = etiqueta.value_counts()
display(pd.DataFrame({"pasajeros": conteo, "porcentaje": (conteo / len(df) * 100).round(1)}))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sns.kdeplot(data=df.assign(estado=etiqueta), x="age", hue="estado", palette=colores, common_norm=False, fill=True, ax=axes[0, 0])
axes[0, 0].set_title("Edad")
sns.histplot(data=df.assign(estado=etiqueta, log_tarifa=np.log1p(df["fare"])), x="log_tarifa", hue="estado", palette=colores,
             bins=30, ax=axes[0, 1])
axes[0, 1].set_title("Tarifa (con log1p para ver mejor)")
axes[0, 1].set_xlabel("log1p(fare)")
for ax, col, titulo in zip([axes[0, 2], axes[1, 0], axes[1, 1], axes[1, 2]], ["pclass", "sex", "embarked", "sibsp"],
                           ["Clase", "Sexo", "Puerto de embarque", "Hermanos y cónyuges (sibsp)"]):
    tasa = df.groupby(col)["survived"].mean() * 100
    sns.barplot(x=tasa.index.astype(str), y=tasa.values, color="#8172b3", ax=ax)
    ax.set_title(f"% que sobrevivió por {titulo.lower()}" if col != "sibsp" else "% que sobrevivió por sibsp")
    ax.set_xlabel("")
    ax.set_ylabel("% que sobrevivió")
plt.tight_layout()
plt.show()

for col in ["pclass", "sex", "embarked", "sibsp"]:
    tasa = (df.groupby(col)["survived"].agg(["mean", "size"]).rename(columns={"mean": "% que sobrevivió", "size": "pasajeros"}))
    tasa["% que sobrevivió"] = (tasa["% que sobrevivió"] * 100).round(1)
    display(tasa.T)

**Qué vemos.** Sobrevivieron 342 pasajeros (38.4%) y 549 no (61.6%), así que un modelo que dijera siempre "no sobrevive" acertaría el 61.6% sin aprender nada. Ese es el piso contra el que hay que comparar la accuracy.

En las gráficas, lo que más separa a los dos grupos es el sexo, la clase y, en menor medida, la tarifa:

- **Edad:** las dos curvas son parecidas, salvo un bulto de niños pequeños donde sobreviven más.
- **Tarifa:** la mayoría de los pasajeros pagó poco, y quienes pagaron más sobrevivieron con más frecuencia. Hay un grupo pequeño con tarifa 0.
- **Clase y sexo:** la supervivencia baja de primera a tercera clase, y entre las mujeres es mucho más alta que entre los hombres.
- **Puerto de embarque:** el puerto C tiene la tasa más alta. Puede reflejar quiénes embarcaron ahí (más pasajeros de primera clase) y no el puerto en sí.
- **Hermanos y cónyuges (`sibsp`):** viajar solo o con un grupo muy grande se asocia con menos supervivencia que viajar con una o dos personas.

In [ ]:
df_exp = df.assign(grupo_edad=pd.cut(df["age"], [0, 12, 17, 59, 100], labels=["0-12", "13-17", "18-59", "60+"]),
                   viaja_con=np.where(df["sibsp"] + df["parch"] == 0, "solo", "con familia"))
tablas = {
    "sexo y clase": df_exp.pivot_table(index="sex", columns="pclass", values="survived", aggfunc="mean") * 100,
    "grupo de edad": df_exp.groupby("grupo_edad", observed=True)["survived"].agg(["mean", "size"]).rename(columns={"mean": "% sobrevivió", "size": "pasajeros"}),
    "viaja solo o no": df_exp.groupby("viaja_con")["survived"].agg(["mean", "size"]).rename(columns={"mean": "% sobrevivió", "size": "pasajeros"}),
}
tablas["grupo de edad"]["% sobrevivió"] *= 100
tablas["viaja solo o no"]["% sobrevivió"] *= 100
for nombre, tabla in tablas.items():
    print("Supervivencia por", nombre)
    display(tabla.round(1))

**Qué vemos.** El sexo y la clase juntos separan casi todo: entre las mujeres sobrevivió el 96.8% de primera clase y el 92.1% de segunda, pero solo el 50.0% de tercera. Entre los hombres, el 36.9% de primera, el 15.7% de segunda y el 13.5% de tercera. Los niños de 0 a 12 años sobrevivieron más (58.0%) que los adultos de 18 a 59 (38.6%), y quienes viajaban con familia más que quienes viajaban solos (50.6% contra 30.4%), aunque esta última diferencia se mezcla con el sexo y la clase.

Es útil tener esto en mente para la comparación: un árbol puede combinar el sexo con la clase en dos preguntas seguidas, y KNN tiene que lograrlo con distancias, así que el escalado va a pesar mucho.

## 2. Train y test, y la validación cruzada que usan los dos modelos

Separamos 80% para train y 20% para test con `stratify=y`, igual que en el proyecto 1 y en las otras actividades del capítulo. Así los dos modelos se comparan sobre los mismos pasajeros. El test no se toca hasta la sección 8.

Para decidir usamos una validación cruzada estratificada de 5 partes sobre train, con `shuffle=True` y `random_state=42`. Es **un solo objeto `cv`** que se pasa a los dos modelos, así los dos ven exactamente las mismas particiones. Las decisiones donde las diferencias son pequeñas se revisan con una versión repetida 10 veces (50 evaluaciones).

In [ ]:
X = df[VARIABLES]
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_repetida = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Supervivencia en train (%):", round(y_train.mean() * 100, 2))
print("Supervivencia en test (%):", round(y_test.mean() * 100, 2))

referencia = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Referencia (siempre 'no sobrevive') en test:", round(referencia.score(X_test, y_test), 4))

Train queda con 712 pasajeros y test con 179, con 38.34% y 38.55% de supervivientes, igual que en las otras actividades. La referencia de siempre "no sobrevive" acierta el 0.6145 en test, y cualquier modelo tiene que superarla para decir que aprendió algo.

Con 179 pasajeros, una accuracy cercana a 0.8 tiene un error estándar de unos 3 puntos. Por eso, si dos modelos quedan a menos de ese margen en test, no se puede decir que uno sea mejor con un solo reparto.

## 3. Preprocesamiento

Hay que decidir cuatro cosas, siempre dentro de un `Pipeline` para que cada paso aprenda solo con los datos de entrenamiento de cada partición:

- **Edad faltante:** se rellena con la mediana de train.
- **Embarque faltante** (solo faltan 2 pasajeros): se rellena con el puerto más frecuente de train.
- **Sexo y puerto:** son categorías, así que se codifican con `OneHotEncoder` (`drop="if_binary"` deja una sola columna para `sex`).
- **Escala:** la única decisión que no es evidente. A un árbol le da igual, pero KNN suma diferencias entre columnas, y no es lo mismo una diferencia de 10 libras de tarifa que de 1 clase.

Para la escala comparamos tres opciones con KNN y `k` de 1, 5 y 15, con la validación repetida: no escalar, escalar solo las columnas numéricas, y escalar también las columnas de sexo y puerto (las que salen del one-hot). Escribimos la regla antes de ver el resultado: si dos opciones quedan a menos de 0.01 de accuracy en promedio, se queda la más simple.

In [ ]:
NUMERICAS = ["pclass", "age", "sibsp", "parch", "fare"]
CATEGORICAS = ["sex", "embarked"]


def armar_preprocesador(escalar="numericas", log_tarifa=False):
    """escalar: 'ninguno', 'numericas' o 'todo'."""
    numericas = [c for c in NUMERICAS if not (log_tarifa and c == "fare")]
    pasos_num = [("imputador", SimpleImputer(strategy="median"))]
    pasos_cat = [("imputador", SimpleImputer(strategy="most_frequent")),
                 ("codificador", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False))]
    if escalar in ("numericas", "todo"):
        pasos_num.append(("escalador", StandardScaler()))
    if escalar == "todo":
        pasos_cat.append(("escalador", StandardScaler()))
    bloques = [("num", Pipeline(pasos_num), numericas), ("cat", Pipeline(pasos_cat), CATEGORICAS)]
    if log_tarifa:
        bloques.append(("tarifa", Pipeline([
            ("imputador", SimpleImputer(strategy="median")),
            ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("escalador", StandardScaler()),
        ]), ["fare"]))
    return ColumnTransformer(bloques)


def armar_knn(k, **opciones):
    return Pipeline([("preprocesador", armar_preprocesador(**opciones)), ("knn", KNeighborsClassifier(n_neighbors=k))])


def armar_arbol(**parametros):
    return Pipeline([("preprocesador", armar_preprocesador("ninguno")),
                     ("arbol", DecisionTreeClassifier(random_state=SEED, **parametros))])


opciones_escala = {
    "No escalar": dict(escalar="ninguno"),
    "Escalar las numéricas": dict(escalar="numericas"),
    "Escalar todo (también sexo y puerto)": dict(escalar="todo"),
    "Escalar las numéricas y log1p en tarifa": dict(escalar="numericas", log_tarifa=True),
}
filas = {}
for nombre, opciones in opciones_escala.items():
    filas[nombre] = {f"k={k}": cross_val_score(armar_knn(k, **opciones), X_train, y_train, cv=cv_repetida).mean() for k in (1, 5, 15)}
tabla_escala = pd.DataFrame(filas).T
tabla_escala["promedio"] = tabla_escala.mean(axis=1)
display(tabla_escala.round(4))

MARGEN_ESCALA = 0.01
cercanas = tabla_escala[tabla_escala["promedio"] >= tabla_escala["promedio"].max() - MARGEN_ESCALA]
print("Opciones a menos de", MARGEN_ESCALA, "del mejor promedio:", list(cercanas.index))

**Qué vemos.** Escalar hace una diferencia grande: sin escalar, KNN promedia 0.7039 de accuracy en validación, y escalando las numéricas llega a 0.7813. Con `k = 5` pasa de 0.7134 a 0.7928.

Tres opciones quedan a menos de 0.01 del mejor promedio (escalar solo las numéricas, escalar todo y escalar con `log1p` en la tarifa). Escalar todo tiene el promedio más alto (0.7886), pero la diferencia contra escalar solo las numéricas es de 0.0073, menor que el margen que escribimos. Por la regla nos quedamos con la más simple: **escalar solo las columnas numéricas**.

En el árbol escalar no cambia nada: la accuracy de validación es 0.8132 con y sin escalado. El árbol solo pregunta si un valor pasa de un umbral, y el escalado no altera el orden de los pasajeros. Por eso el árbol se entrena sin escalar.

In [ ]:
# El árbol no usa distancias: comprobamos que escalar no le cambia nada
arbol_sin = cross_val_score(armar_arbol(max_depth=4), X_train, y_train, cv=cv)
arbol_con = cross_val_score(Pipeline([("preprocesador", armar_preprocesador("numericas")),
                                      ("arbol", DecisionTreeClassifier(max_depth=4, random_state=SEED))]), X_train, y_train, cv=cv)
print("Árbol (profundidad 4), accuracy en validación sin escalar:", arbol_sin.mean().round(4), "| escalando:", arbol_con.mean().round(4))

## 4. Cómo decide KNN

Antes de elegir `k`, hacemos a mano lo que hace `KNeighborsClassifier`. Para un pasajero nuevo: (1) se calcula su distancia a cada pasajero de train, (2) se toman los `k` más cercanos y (3) se cuenta cuántos sobrevivieron. La distancia es la euclidiana, $\sqrt{\sum (a_i - b_i)^2}$, sobre las columnas ya preprocesadas.

Lo hacemos con el primer pasajero de test, `k = 5` y la escala que salió de la sección 3, y lo comparamos con scikit-learn.

In [ ]:
PREPROCESO_ELEGIDO = dict(escalar="numericas")

preprocesador = armar_preprocesador(**PREPROCESO_ELEGIDO).fit(X_train)
Z_train = preprocesador.transform(X_train)
pasajero = X_test.iloc[[0]]
z = preprocesador.transform(pasajero).iloc[0]

k = 5
distancias = np.sqrt(((Z_train - z) ** 2).sum(axis=1))
vecinos = distancias.sort_values().index[:k]

detalle = X_train.loc[vecinos].assign(distancia=distancias[vecinos].round(3), sobrevivió=y_train.loc[vecinos])
print("Pasajero de test:")
display(pasajero)
print(f"Sus {k} vecinos más cercanos en train:")
display(detalle)

votos = y_train.loc[vecinos].mean()
print(f"Sobrevivieron {int(votos * k)} de {k}: probabilidad a mano = {votos}")

modelo_k5 = armar_knn(k, **PREPROCESO_ELEGIDO).fit(X_train, y_train)
print("predict_proba de scikit-learn:", modelo_k5.predict_proba(pasajero)[0, 1], "| predicción:", modelo_k5.predict(pasajero)[0],
      "| realidad:", int(y_test.iloc[0]))

**Qué vemos.** El pasajero 565 de test es un hombre de tercera clase de 24 años, con 2 hermanos o cónyuges y una tarifa de 24.15. Sus cinco vecinos más cercanos en train son otros hombres de tercera clase que embarcaron en Southampton, con distancias de 0.357 a 1.009 (uno de ellos tenía la edad vacía y entra con la mediana, como indica el preprocesamiento). De los cinco, solo uno sobrevivió.

Con un voto de 1 contra 4, la probabilidad a mano es 0.2 y la predicción es "no sobrevive". Es exactamente lo que devuelve `predict_proba` de scikit-learn, y coincide con lo que le pasó al pasajero. Esa es toda la lógica de KNN: no ajusta ninguna fórmula, solo guarda los datos de train y vota entre los vecinos.

In [ ]:
# ¿Cuánto pesa cada columna en la distancia si NO se escala?
sin_escalar = armar_preprocesador("ninguno").fit_transform(X_train)
varianzas = sin_escalar.var()
aporte = (varianzas / varianzas.sum() * 100).sort_values(ascending=False)
print("Porcentaje de la distancia al cuadrado (en promedio) que aporta cada columna sin escalar:")
display(aporte.round(2).rename("% de la distancia"))

**Qué vemos.** Sin escalar, la tarifa aporta el 93.04% de la distancia al cuadrado entre dos pasajeros y la edad el 6.83%. Las otras siete columnas (clase, hermanos, padres, sexo y puerto) suman menos de 0.2%.

Esto explica el resultado de la sección 3. La tarifa va de 0 a 512 libras y la clase de 1 a 3, así que una diferencia de 10 libras pesa mucho más que cambiar de clase, aunque la clase y el sexo sean las dos variables que más informan sobre la supervivencia. Sin escalar, KNN prácticamente ignora justo lo que más ayudaría. Al estandarizar, cada columna numérica queda con media 0 y desviación 1, y todas pesan en la misma medida.

## 5. Árbol de decisión

Primero un árbol sin límites, para ver qué pasa. Después `GridSearchCV` sobre la validación cruzada `cv` con `max_depth` de 1 a 10 y `min_samples_leaf` de 1, 5, 10 y 20.

In [ ]:
arbol_libre = armar_arbol().fit(X_train, y_train)
validacion_libre = cross_val_score(armar_arbol(), X_train, y_train, cv=cv)
print("Árbol sin límites -> hojas:", arbol_libre.named_steps["arbol"].get_n_leaves(),
      "| profundidad:", arbol_libre.named_steps["arbol"].get_depth())
print("accuracy en train:", round(arbol_libre.score(X_train, y_train), 4), "| en validación:", validacion_libre.mean().round(4))

busqueda_arbol = GridSearchCV(armar_arbol(), {"arbol__max_depth": list(range(1, 11)), "arbol__min_samples_leaf": [1, 5, 10, 20]},
                              cv=cv, scoring="accuracy", return_train_score=True)
busqueda_arbol.fit(X_train, y_train)

resultados = pd.DataFrame(busqueda_arbol.cv_results_)
resultados["hojas"] = [
    armar_arbol(max_depth=p["arbol__max_depth"], min_samples_leaf=p["arbol__min_samples_leaf"]).fit(X_train, y_train).named_steps["arbol"].get_n_leaves()
    for p in resultados["params"]
]
mejores = resultados.sort_values(["mean_test_score", "hojas"], ascending=[False, True])[
    ["param_arbol__max_depth", "param_arbol__min_samples_leaf", "hojas", "mean_train_score", "mean_test_score", "std_test_score"]
].head(6)
mejores.columns = ["max_depth", "min_samples_leaf", "hojas", "accuracy train", "accuracy validación", "desviación"]
display(mejores.round(4).reset_index(drop=True))

arbol_final = busqueda_arbol.best_estimator_
print("Elegido:", busqueda_arbol.best_params_, "| hojas:", arbol_final.named_steps["arbol"].get_n_leaves())

In [ ]:
profundidades = resultados[resultados["param_arbol__min_samples_leaf"] == 1].sort_values("param_arbol__max_depth")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(profundidades["param_arbol__max_depth"], profundidades["mean_train_score"], marker="o", label="train")
ax.plot(profundidades["param_arbol__max_depth"], profundidades["mean_test_score"], marker="o", label="validación")
ax.axvline(busqueda_arbol.best_params_["arbol__max_depth"], color="gray", linestyle="--")
ax.set_xlabel("max_depth (con min_samples_leaf=1)")
ax.set_ylabel("Accuracy")
ax.set_title("Árbol: train contra validación")
ax.legend()
plt.show()

print("Accuracy de train con max_depth=10 y min_samples_leaf=1:", round(float(profundidades["mean_train_score"].iloc[-1]), 4))
nombres = arbol_final.named_steps["preprocesador"].get_feature_names_out()
print(export_text(arbol_final.named_steps["arbol"], feature_names=list(nombres), max_depth=3))
importancias = pd.Series(arbol_final.named_steps["arbol"].feature_importances_, index=nombres).sort_values(ascending=False)
importancias[importancias > 0].round(3)

**Qué vemos.**

- **El árbol sin límites memoriza.** Tiene 152 hojas y 23 niveles de profundidad, acierta el 0.9831 en train y solo el 0.7753 en validación.
- **Limitarlo ayuda.** `GridSearchCV` elige `max_depth = 4` y `min_samples_leaf = 1`, con 15 hojas: 0.8459 en train y 0.8132 en validación (desviación de 0.0134). La gráfica lo muestra: la accuracy de train sube con la profundidad hasta 0.9431 con 10 niveles, mientras la de validación llega a su máximo en 4 y después baja.
- **Hay varios casi empates.** Las seis mejores combinaciones quedan entre 0.8090 y 0.8132. Una de profundidad 7 y 40 hojas llega al mismo 0.8132, pero conviene quedarse con la de 15 hojas, que es mucho más fácil de explicar.

Las reglas del árbol elegido empiezan con el sexo (que es también la variable con más importancia, 0.576, seguida de la clase con 0.196 y la edad con 0.107). Entre las mujeres, las de primera y segunda clase se predicen como sobrevivientes, salvo una banda muy estrecha de tarifa que parece más ruido del train que una regla del barco. Las de tercera clase sobreviven si no embarcaron en Southampton y tienen 29.75 años o menos. Entre los hombres, el árbol solo predice supervivencia para algunos niños de 3.5 años o menos; todos los de más edad caen en hojas de "no sobrevive", varias de ellas con divisiones por tarifa que no cambian la predicción.

Esto anticipa un resultado: como el árbol casi nunca predice "sobrevive" para un hombre, es difícil que detecte a todos los sobrevivientes.

## 6. KNN: elegir `k`

Para cada `k` de 1 a 30 medimos dos cosas con el preprocesamiento elegido:

- **accuracy de train**: el modelo entrenado con todo train y evaluado sobre esos mismos pasajeros;
- **accuracy de validación**: la media sobre las 5 particiones de `cv`, donde cada parte se evalúa con un modelo que no la vio.

El `k` elegido es el que alcanza el máximo en validación.

In [ ]:
filas = []
for k in range(1, 31):
    modelo = armar_knn(k, **PREPROCESO_ELEGIDO)
    filas.append({
        "k": k,
        "train": modelo.fit(X_train, y_train).score(X_train, y_train),
        "validación": cross_val_score(modelo, X_train, y_train, cv=cv).mean(),
        "validación (repetida)": cross_val_score(modelo, X_train, y_train, cv=cv_repetida).mean(),
    })
curva = pd.DataFrame(filas).set_index("k")

mejor_k = int(curva["validación"].idxmax())
mejor_k_repetida = int(curva["validación (repetida)"].idxmax())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(curva.index, curva["train"], marker="o", label="train (los mismos pasajeros del entrenamiento)")
ax.plot(curva.index, curva["validación"], marker="o", label="validación (5 partes)")
ax.axvline(mejor_k, color="gray", linestyle="--", label=f"k elegido = {mejor_k}")
ax.axhline(y_train.pipe(lambda s: 1 - s.mean()), color="red", linestyle=":", label="siempre 'no sobrevive'")
ax.set_xlabel("k (número de vecinos)")
ax.set_ylabel("Accuracy")
ax.set_title("KNN: accuracy de train y de validación contra k")
ax.legend()
plt.show()

print("k elegido (validación con cv):", mejor_k, "| accuracy:", round(curva.loc[mejor_k, "validación"], 4))
print("k que sale con la validación repetida:", mejor_k_repetida, "| accuracy:", round(curva.loc[mejor_k_repetida, "validación (repetida)"], 4))
curva.loc[[1, 2, 3, 5, 10, 15, 20, mejor_k, 25, 30]].round(4)

In [ ]:
# ¿Por qué con k=1 la accuracy de train no es 1?
Z_train = armar_preprocesador(**PREPROCESO_ELEGIDO).fit_transform(X_train)
grupos = Z_train.assign(y=y_train).groupby(list(Z_train.columns))["y"].agg(["size", "nunique"])
repetidos = int(grupos.loc[grupos["size"] > 1, "size"].sum())
en_conflicto = int(grupos.loc[grupos["nunique"] > 1, "size"].sum())

print("Pasajeros de train con otro pasajero idéntico después del preprocesamiento:", repetidos, "de", len(Z_train))
print("De ellos, los que comparten ese perfil con alguien que tiene el resultado contrario:", en_conflicto)
print("Accuracy de train con k=1:", round(curva.loc[1, "train"], 4))

**Qué vemos.**

- **Con `k = 1` el modelo memoriza.** La accuracy de train es 0.9761 y la de validación solo 0.7500. Cada pasajero de train es su propio vecino más cercano, así que acierta casi siempre sobre los mismos datos con los que se entrenó, pero con un pasajero nuevo depende de un solo vecino, que puede ser ruido.
- **Al subir `k` el modelo se suaviza y las dos curvas se acercan.** La accuracy de train baja (0.8792 con `k = 2` y alrededor de 0.82 a 0.83 desde `k = 10`) y la de validación sube (de 0.7880 con `k = 3` a 0.8160 con `k = 20`).
- **El máximo en validación está en `k = 21`**, con 0.8202 (train 0.8315). Después la curva baja despacio: con `k = 30` la validación es 0.7978, porque con demasiados vecinos votan pasajeros que ya no se parecen al nuevo.
- **El máximo es poco firme.** Con la validación repetida el mejor valor sale en `k = 18` (0.8126), y la curva se mantiene casi plana entre `k = 14` y `k = 23`. Elegimos 21 porque así lo pide la consigna (el máximo con `cv`), pero cualquier `k` de esa zona daría un resultado equivalente.

La accuracy de train con `k = 1` no llega a 1 porque hay pasajeros con exactamente el mismo perfil después del preprocesamiento: 130 de los 712 comparten perfil con otro, y 36 de ellos con alguien que tiene el resultado contrario. En esos casos el único vecino no puede coincidir con los dos.

## 7. Los dos modelos, en validación y en test

Esta es la primera vez que usamos el test. Los dos modelos ya están elegidos, uno con `GridSearchCV` y otro con la curva de `k`, y no se cambian con lo que salga aquí. La tabla incluye la referencia sencilla: predecir siempre la clase mayoritaria de train.

In [ ]:
knn_final = armar_knn(mejor_k, **PREPROCESO_ELEGIDO).fit(X_train, y_train)

modelos = {
    f"Árbol (depth={busqueda_arbol.best_params_['arbol__max_depth']}, leaf={busqueda_arbol.best_params_['arbol__min_samples_leaf']})": arbol_final,
    f"KNN (k={mejor_k})": knn_final,
    "Referencia: siempre 'no sobrevive'": referencia,
}

filas = []
for nombre, modelo in modelos.items():
    validacion = cross_val_score(modelo, X_train, y_train, cv=cv)
    pred = modelo.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()      # clase 0 = no sobrevivió, clase 1 = sobrevivió
    filas.append({
        "modelo": nombre,
        "validación (media)": validacion.mean(),
        "validación (desviación)": validacion.std(),
        "test": modelo.score(X_test, y_test),
        "aciertos en test (de %d)" % len(y_test): int((pred == y_test).sum()),
        "recall de sobrevivientes": recall_score(y_test, pred, zero_division=0),
        "precision de sobrevivientes": precision_score(y_test, pred, zero_division=0),
        "sobrevivientes que predice que no": int(fn),
        "no sobrevivientes que predice que sí": int(fp),
    })
tabla_final = pd.DataFrame(filas).set_index("modelo")
display(tabla_final.round(4))
print("Error estándar de una accuracy cercana a 0.8 con", len(y_test), "pasajeros:", round(np.sqrt(0.8 * 0.2 / len(y_test)), 3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (nombre, modelo) in zip(axes, modelos.items()):
    matriz = confusion_matrix(y_test, modelo.predict(X_test))
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["no sobrevive", "sobrevive"], yticklabels=["no sobrevivió", "sobrevivió"])
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

**Qué vemos.**

- **Validación:** KNN con `k = 21` saca 0.8202 (desviación 0.0227) y el árbol 0.8132 (desviación 0.0134). La diferencia de 0.007 es menor que la variación entre particiones, así que están empatados.
- **Test:** el árbol acierta 141 de 179 (0.7877) y KNN 137 de 179 (0.7654). Son 4 pasajeros de diferencia, menos que el error estándar de 3 puntos, así que tampoco se puede afirmar que uno sea mejor con este reparto. Los dos superan con claridad a la referencia, que acierta 110 de 179 (0.6145): más de 15 puntos de ventaja.
- **Los errores son de tipo distinto.** En test hay 69 pasajeros que sobrevivieron. El árbol deja de detectar a 31 de ellos y solo marca como sobrevivientes a 7 que no lo eran (precision 0.8444, recall 0.5507). KNN deja de detectar a 26 y marca de más a 16 (precision 0.7288, recall 0.6232). El árbol es más cauteloso: dice "sobrevive" pocas veces, pero casi siempre acierta. KNN dice "sobrevive" más seguido y se equivoca más veces al hacerlo.

En los dos modelos el error principal son los sobrevivientes que no se detectan. En el árbol se explica en parte por sus reglas: para todos los hombres de más de 3.5 años predice "no sobrevive" (sección 5), así que no puede acertar con los que sí sobrevivieron.

## 8. ¿Qué tan estable es la comparación?

Con 179 pasajeros en test, una sola comparación puede favorecer a cualquiera de los dos. Repetimos todo con 20 repartos distintos (`random_state` de 0 a 19). En cada uno separamos, elegimos los hiperparámetros solo con el train de ese reparto (`k` de 1 a 30 para KNN; profundidad y hojas mínimas para el árbol) y medimos en su test. Esto no sirve para escoger el modelo; solo mide la variación.

In [ ]:
filas = []
for semilla in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=semilla)
    cv_local = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    knn_local = GridSearchCV(armar_knn(5, **PREPROCESO_ELEGIDO), {"knn__n_neighbors": list(range(1, 31))}, cv=cv_local).fit(X_tr, y_tr)
    arbol_local = GridSearchCV(armar_arbol(), {"arbol__max_depth": list(range(1, 11)), "arbol__min_samples_leaf": [1, 5, 10, 20]},
                               cv=cv_local).fit(X_tr, y_tr)
    filas.append({
        "reparto": semilla,
        "KNN": knn_local.score(X_te, y_te),
        "k elegido": knn_local.best_params_["knn__n_neighbors"],
        "Árbol": arbol_local.score(X_te, y_te),
        "profundidad elegida": arbol_local.best_params_["arbol__max_depth"],
        "Referencia": (y_te == 0).mean(),
    })
repartos = pd.DataFrame(filas).set_index("reparto")

resumen = repartos[["KNN", "Árbol", "Referencia"]].agg(["mean", "min", "max"]).T
resumen.columns = ["media", "mínimo", "máximo"]
display(resumen.round(4))

diferencia = repartos["Árbol"] - repartos["KNN"]
print("Reparto 42 (el de este notebook): árbol", round(arbol_final.score(X_test, y_test), 4), "| KNN", round(knn_final.score(X_test, y_test), 4))
print("Gana el árbol en", int((diferencia > 0).sum()), "repartos, gana KNN en", int((diferencia < 0).sum()), "y empatan en", int((diferencia == 0).sum()))
print("Diferencia media (árbol menos KNN):", round(diferencia.mean(), 4))
print("k elegido por KNN: de", repartos["k elegido"].min(), "a", repartos["k elegido"].max(), "| mediana", int(repartos["k elegido"].median()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(repartos.index, repartos["Árbol"], marker="o", label="Árbol")
ax.plot(repartos.index, repartos["KNN"], marker="o", label="KNN")
ax.axhline(repartos["Referencia"].mean(), color="red", linestyle=":", label="Referencia")
ax.set_xlabel("Reparto (random_state)")
ax.set_ylabel("Accuracy en test")
ax.set_title("Árbol y KNN en 20 repartos distintos")
ax.set_xticks(range(0, 20, 2))
ax.legend()
plt.show()

**Qué vemos.**

- **El árbol promedia 0.8193** (de 0.7709 a 0.8547) **y KNN 0.7994** (de 0.7654 a 0.8492). Los dos superan con claridad la referencia de 0.6145 en todos los repartos.
- **El árbol gana en 15 repartos, KNN en 3 y empatan en 2**, con una diferencia media de 0.0198 a favor del árbol, un poco menos de 2 puntos. Es una ventaja pequeña, aunque se repite en la mayoría de los repartos. Hay que tomarla con cuidado, porque los 20 repartos comparten pasajeros y no son pruebas independientes.
- **Nuestro reparto 42 salió bajo para los dos.** El árbol saca 0.7877, contra un promedio de 0.8193, y KNN saca 0.7654, que es justo su mínimo en los 20 repartos. En promedio, con otros repartos los dos salen mejor.
- **El `k` elegido por KNN cambia mucho** de un reparto a otro, de 4 a 26 con mediana de 18. Esto confirma que la curva de validación es casi plana en esa zona y que no hay un `k` mejor que los demás.

## 9. Extensión: ¿ayudan las variables del proyecto 1?

La consigna pide los dos modelos con las siete variables de la tabla, y eso es lo que usamos en las secciones anteriores. Como comparación extra, probamos las variables que armamos en la actividad del árbol. Salen de esas mismas siete columnas, fila por fila, sin usar nada del test ni de `alive`:

- `family_size = sibsp + parch + 1`: personas del grupo, contando al pasajero.
- `fare_por_persona = fare / family_size`: lo que pagó cada persona, porque el boleto cubre a todo el grupo.
- `es_mujer_o_nino`: 1 si es mujer o tiene 12 años o menos (la norma de abordaje de los botes). Si la edad falta, cuenta como no niño.

Con ellas formamos el conjunto de cuatro variables `es_mujer_o_nino`, `pclass`, `fare_por_persona` y `family_size`. Para no usar el test dos veces, esta comparación se hace **solo con validación cruzada repetida sobre train**.

In [ ]:
def agregar_variables(datos):
    datos = datos.copy()
    datos["family_size"] = datos["sibsp"] + datos["parch"] + 1
    datos["fare_por_persona"] = datos["fare"] / datos["family_size"]
    datos["es_mujer_o_nino"] = ((datos["sex"] == "female") | (datos["age"] <= 12)).astype(int)
    return datos


X_train_ext = agregar_variables(X_train)
CUATRO = ["es_mujer_o_nino", "pclass", "fare_por_persona", "family_size"]


def pipeline_cuatro(modelo, escalar):
    pasos = [("imputador", SimpleImputer(strategy="median"))] + ([("escalador", StandardScaler())] if escalar else [])
    return Pipeline([("preprocesador", ColumnTransformer([("num", Pipeline(pasos), CUATRO)])), ("modelo", modelo)])


filas = []
for nombre, variables in {"Las 7 variables de la tabla": "siete", "Las 4 variables del proyecto 1": "cuatro"}.items():
    if variables == "siete":
        arbol = armar_arbol(**{k.split("__")[1]: v for k, v in busqueda_arbol.best_params_.items()})
        knn = armar_knn(mejor_k, **PREPROCESO_ELEGIDO)
        datos = X_train
    else:
        arbol = pipeline_cuatro(DecisionTreeClassifier(random_state=SEED, **{k.split("__")[1]: v for k, v in busqueda_arbol.best_params_.items()}), False)
        knn = pipeline_cuatro(KNeighborsClassifier(n_neighbors=mejor_k), True)
        datos = X_train_ext
    filas.append({
        "variables": nombre,
        "Árbol": cross_val_score(arbol, datos, y_train, cv=cv_repetida).mean(),
        f"KNN (k={mejor_k})": cross_val_score(knn, datos, y_train, cv=cv_repetida).mean(),
    })
display(pd.DataFrame(filas).set_index("variables").round(4))

# KNN con las 4 variables: ¿cambia el mejor k?
valores_k = list(range(1, 31))
curva_cuatro = pd.Series({k: cross_val_score(pipeline_cuatro(KNeighborsClassifier(n_neighbors=k), True), X_train_ext, y_train, cv=cv_repetida).mean()
                          for k in valores_k})
print("Con las 4 variables, el mejor k en validación repetida es", int(curva_cuatro.idxmax()), "con accuracy", round(curva_cuatro.max(), 4))

**Qué vemos.** Con las siete variables de la tabla, el árbol saca 0.8171 y KNN 0.8119 en validación repetida. Con las cuatro variables del proyecto 1, el árbol sube a 0.8367 (casi 2 puntos más) y KNN se queda en 0.8121, prácticamente igual. Para KNN con las cuatro variables, el mejor `k` en validación repetida pasa a 27, con 0.8159.

Una posible razón es que el árbol necesita que le armemos las combinaciones: no puede sumar `sibsp` y `parch` ni juntar el sexo con la edad por su cuenta. No comprobamos esa explicación. Lo que sí vimos es que estas variables ayudan al árbol y no a KNN.

Como esta comparación no usa el test, los números de las secciones 7 y 8, con las siete variables de la consigna, siguen siendo los que se reportan.

## Conclusiones

**Con qué nos quedamos.** Con el **árbol de decisión** (`max_depth = 4`, 15 hojas), con las siete variables de la tabla. La ventaja sobre KNN es pequeña, pero se sostiene en tres lugares: acierta 141 de 179 en test contra 137, gana en 15 de 20 repartos con una diferencia media de 0.0198, y varía menos entre particiones (desviación de 0.0134 contra 0.0227). Además se puede explicar con una lista de reglas, no necesita escalar y no depende de un `k` que cambia de un reparto a otro.

**Decisiones y evidencia**

| Decisión | Qué se hizo | Evidencia (solo train) |
|---|---|---|
| Variables | Las 7 de la tabla; `alive` y las demás columnas extra no se usan | `alive` repite `survived` sin excepciones (549 y 342) |
| Valores faltantes | Mediana en `age`, valor más frecuente en `embarked`, dentro del Pipeline | 177 y 2 faltantes; el Pipeline los rellena solo con train |
| Escalar | Sí para KNN (solo las numéricas), no para el árbol | KNN pasa de 0.7039 a 0.7813; el árbol queda en 0.8132 con y sin escalar |
| Profundidad del árbol | `max_depth = 4`, `min_samples_leaf = 1` | `GridSearchCV`: 0.8132 en validación, contra 0.7753 del árbol sin límites |
| `k` de KNN | `k = 21` | Máximo de la validación (0.8202); con la validación repetida sale 18, y la curva es casi plana |
| Modelo final | Árbol | Test 0.7877 contra 0.7654, y 15 de 20 repartos |

**Train contra validación con `k = 1`.** Con `k = 1` la accuracy de train es 0.9761 y la de validación 0.7500: el modelo copia a su vecino y memoriza. Al aumentar `k` las dos curvas se acercan, y con `k = 21` quedan en 0.8315 y 0.8202. Con `k` muy grande la validación vuelve a bajar (0.7978 con `k = 30`).

**Qué errores comete cada modelo.** Los dos fallan sobre todo con los sobrevivientes. El árbol deja de detectar a 31 de los 69 sobrevivientes de test y solo se equivoca con 7 de los que predice como sobrevivientes. KNN deja de detectar a 26 y se equivoca con 16. Si el objetivo fuera encontrar a la mayor cantidad posible de sobrevivientes, KNN sería la mejor opción (recall de 0.6232 contra 0.5507), a costa de más falsas alarmas.

**Limitaciones**

- Con 179 pasajeros en test, el error estándar es de unos 3 puntos, así que las diferencias de 2 y 4 puntos entre los modelos no son concluyentes por sí solas. Por eso la conclusión se apoya en los 20 repartos y no en uno.
- El reparto 42 salió bajo para los dos modelos, y el 0.7654 de KNN es su mínimo entre los 20.
- Elegimos los hiperparámetros con la misma validación cruzada con la que reportamos, así que esos números son algo optimistas.
- Son 891 pasajeros de un solo barco, y con las siete variables de la tabla ninguno de los dos detecta bien a los sobrevivientes que no encajan en el patrón de sexo y clase.

**Qué haríamos con más tiempo:** medir todo el proceso, incluida la elección de `k` y de la profundidad, con validación cruzada anidada, y probar KNN con otra métrica de distancia o con pesos por cercanía. Con las variables del proyecto 1 el árbol llega a 0.8367 en validación repetida, un camino que valdría la pena confirmar con el test si la consigna lo permitiera.